**RAG Debugging**

In [74]:
import os
import time
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from huggingface_hub import get_collection
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google import genai
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
load_dotenv()

True

In [75]:
parser = StrOutputParser()

In [76]:
#all api keys

#QDRANT
QDRANT_API_KEY = os.getenv("QDRANTAPIKEY")
QDRANT_ENDPOINT = os.getenv("QDRANTENDPOINT")

#gemini model apikey
geminiapikey = os.getenv("GEMINIAPIKEY")
# gemini_llm = genai.Client(api_key=geminiapikey)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", # or whatever gemini model you are using
    api_key=geminiapikey    # Optional if already set in environment variables
)


In [77]:
# Embeddings model
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2177.30it/s]


In [78]:
file_path = "yarvalley.txt"
loader = TextLoader(file_path)
text_file = loader.load()

In [79]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=150,
    separators=[
        "\n\n",   # paragraphs (highest priority)
        "\n",     # lines
        ". ",     # sentences
        " ",      # words
        ""        # fallback
    ]
)

split_document = splitter.split_documents(text_file)


In [80]:
client = QdrantClient(
       url=QDRANT_ENDPOINT,
       api_key=QDRANT_API_KEY

)

In [81]:
collections = client.get_collections().collections


#Any go through list and stop at true(meet your condition)
collection_exist = any(
    collection.name == "ragval"
    for collection in collections
)

if not collection_exist:
    vectorestore = QdrantVectorStore.from_documents(
        documents=split_document,
        api_key=QDRANT_API_KEY,
        url=QDRANT_ENDPOINT,
        embedding=embeddings,
        collection_name="ragval"
    )

else:
    vectorestore = QdrantVectorStore.from_existing_collection(
        collection_name="ragval",
        api_key=QDRANT_API_KEY,
        url=QDRANT_ENDPOINT,
        embedding=embeddings
    )

    print("Collection already exists. Using existing collection.")



Collection already exists. Using existing collection.


In [82]:
retriever = vectorestore.as_retriever(search_kwargs={"k":5})

In [83]:
rag_prompt = ChatPromptTemplate.from_template(
    """
     Please answer the following questions,if it is in context otherwise answer "I don,t have enough
     information about this.

     context:
     {context}

     question:
     {question}
    """
)

In [84]:
#A function that convert the retrieved,chunks of data into a string and remove the metadata
def get_content(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [85]:
rag_chain = (
    {
        "context": retriever | RunnableLambda(get_content),
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | parser  )

In [86]:
question = "What do you know about malam jabba."
answer = rag_chain.invoke(question)
print(answer)

Malam Jabba is a modern hill resort located about 44 km from Mingora in Swat. It is known for its majestic landscape, adorned with snowy mountain peaks, lush green valleys, and sprawling forests, making it a star destination for adventure seekers.

Initially, it housed a winter ski resort that failed over time. However, in September 2015, the TCKP (Tourism Corporation Khyber Pakhtunkhwa) awarded the restoration of a state-of-the-art ski resort, which included a new chairlift and a 3-4 star hotel. A year later, the chairlift and skiing resort were inaugurated, with the hotel construction nearing completion. Malam Jabba offers a cool and serene atmosphere, refreshing minds with its natural beauty.
